# SEN122A Statistical Analysis of Choice Behaviour

## `Lab session 02C:`
## `Error components and randomly distributed taste parameters`

**Delft University of Technology**<br>
**Q2 2026**<br>
**Instructor:** Sander van Cranenburgh<br>
**TA:** Gabriel Nova

### `Instructions`

This is an independent, exam-style lab. You are given a dataset, its context, and a set of tasks. Decide for yourself how to structure the analysis and which code from Labs 02A and 02B you can reuse. The notebook deliberately contains less guidance than the earlier labs. The lab is not graded and does not have to be submitted.

In this lab session, **your task** is to estimate several mixed logit models with different specifications and random distributions and compare their fit and performance. You will also be asked to interpret the results and discuss the implications of your findings. 

**`Learning objectives lab session 02C`**

After completing the following lab session, you will be able to:
* Independently estimate and interpret mixed logit models, including:
  * Data preparation and cleaning for mixed logit analysis
  * Estimate Error Components mixed logit models
  * Estimate mixed logit models with both error components and random parameters
  * Test different random distributions for the random parameters
* Interpret the results of mixed logit models and discuss their implications for planning and policy.

**`This lab consists of 5 parts:`**

**Part 1**: Load and explore the data set<br>
**Part 2**: Linear additive RUM-MNL models<br>
**Part 3**: Error Components mixed logit models<br>
**Part 4**: Mixed logit models with normally distributed random parameters<br>
**Part 5**: Mixed logit models with negative lognormally distributed random parameters<br>

### `Use of AI tools`
AI tools, such as ChatGPT and GitHub Copilot, are powerful tools to assist with programming. Moreover, in your future careers, you will work in a world where such tools are widely available. As such, we **encourage** you to learn to use AI tools **effectively**.

However, remember that the **primary objective of these lab sessions is not to complete the assignments, but to learn and develop your own understanding and skills**. AI tools can help you achieve this, but they can also undermine your learning if you rely on them too heavily, see this [article](../../Assets/chatbots-and-thinking-english.pdf). Being able to instruct an AI tool to solve a problem is not the same as understanding how to solve it yourself. Understanding requires some effort and friction. You should therefore remain actively involved in conceptualising problems, structuring analyses, interpreting results, and critically evaluating AI-generated solutions.

We recommend using AI tools **carefully**, particularly because you will not have access to them during the `practical exam` (as internet access will be restricted). **Therefore, at all times, make sure you understand what you are doing, not just that your code works.**

### `Documentation hub`
During the exam, you will have access to the **Documentation hub**. This hub contains course materials and other useful resources, such as the lab session solutions and formula cheat sheets.

## Dataset and context

You will use mode choice data from a UK travel survey (`apollo_modeChoice.csv`), which has Stated Preference (SP) data for travel between two UK cities. In each choice task, respondents chose among up to four travel modes: car, bus, air, and rail. The alternatives are described by travel time, travel cost, access time, and service quality.

| Variable | Description | Values |
|---|---|---|
| `ID` | Unique individual ID | 1 to 500 |
| `SP_task` | Index for SP observations | 1 to 14, `NA` for RP |
| `av_car` | Availability for alternative 1 (car) | `1` for available, `0` for unavailable |
| `av_bus` | Availability for alternative 2 (bus) | `1` for available, `0` for unavailable |
| `av_air` | Availability for alternative 3 (air) | `1` for available, `0` for unavailable |
| `av_rail` | Availability for alternative 4 (rail) | `1` for available, `0` for unavailable |
| `time_car` | Travel time (mins) for alternative 1 (car) | Min: 250, mean: 311.79, max: 390 (0 if not available) |
| `cost_car` | Travel cost (£) for alternative 1 (car) | Min: 30, mean: 39.99, max: 50 (0 if not available) |
| `time_bus` | Travel time (mins) for alternative 2 (bus) | Min: 300, mean: 370.29, max: 420 (0 if not available) |
| `cost_bus` | Travel cost (£) for alternative 2 (bus) | Min: 15, mean: 25.02, max: 35 (0 if not available) |
| `access_bus` | Access time (mins) for alternative 2 (bus) | Min: 5, mean: 15.02, max: 25 (0 if not available) |
| `time_air` | Travel time (mins) for alternative 3 (air) | Min: 50, mean: 70.07, max: 90 (0 if not available) |
| `cost_air` | Travel cost (£) for alternative 3 (air) | Min: 50, mean: 79.94, max: 110 (0 if not available) |
| `access_air` | Access time (mins) for alternative 3 (air) | Min: 35, mean: 45.02, max: 55 (0 if not available) |
| `service_air` | Service quality for alternative 3 (air) | `1` for no-frills, `2` for wifi, `3` for food |
| `time_rail` | Travel time (mins) for alternative 4 (rail) | Min: 120, mean: 142.93, max: 170 (0 if not available) |
| `cost_rail` | Travel cost (£) for alternative 4 (rail) | Min: 35, mean: 55.03, max: 75 (0 if not available) |
| `access_rail` | Access time (mins) for alternative 4 (rail) | Min: 5, mean: 14.96, max: 25 (0 if not available) |
| `service_rail` | Service quality for alternative 4 (rail) | `1` for no-frills, `2` for wifi, `3` for food |
| `female` | Dummy variable for female individuals | `1` for female, `0` otherwise |
| `business` | Dummy variable for business trips | `1` for business trips, `0` otherwise |
| `income` | Income variable (£ per annum) | Min: 15,490, mean: 44,748.27, max: 74,891 |
| `choice` | Choice variable | `1` for car, `2` for bus, `3` for air, `4` for rail |

The file is available at `data/apollo_modeChoice.csv`.

#### Run this cell to import the necessary libraries

In [ ]:
# Do not modify this cell.
import os
import sys
from pathlib import Path
import toml

import numpy as np
import pandas as pd
from scipy.stats import norm
import biogeme.biogeme as bio
import biogeme.biogeme_logging as blog
import biogeme.database as db
from biogeme.expressions import Beta, Variable, bioDraws, exp

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from utils.bio_estimation_fcns import estimate_mnl, estimate_panel_mxl, print_results

pd.set_option('display.max_columns', None)


In [ ]:
# Do not change this code
# We set the seed number in the .toml file to ensure reproducibility
with open('biogeme.toml', 'r') as file:
    tomldata = toml.load(file)

# Modify the number of draws
tomldata['MonteCarlo']['seed'] = 42

# Write the modified data back to the .toml file
with open('biogeme.toml', 'w') as file:
    toml.dump(tomldata, file)
    
# Initialize the logger, if it has not been initialized yet
try:
    logger
except NameError:    
    logger = blog.get_screen_logger(level=blog.INFO)
    print('Logger has been initialised')

### `Part 1: Load and explore the data`

Load the dataset.

In [ ]:
# Create the path to the data file
data_path = Path('data/apollo_modeChoice.csv')


`1` Report the number of observations and respondents in the dataset.

In [ ]:
# Your code here

`2` Report the market share of each alternative.

In [ ]:
# Your code here

### `Part 2: Linear-additive RUM-MNL model`

Estimate **Model 1**, a linear-additive RUM-MNL model with generic parameters for travel time, travel cost, and access time. Include alternative-specific constants and service quality dummy variables. Although respondents complete multiple choice tasks, this benchmark model assumes that the unobserved utility terms are independent across tasks.

$$V_{car}=asc_{car}+\beta_{time}time_{car}+\beta_{cost}cost_{car}$$
$$V_{bus}=asc_{bus}+\beta_{time}time_{bus}+\beta_{cost}cost_{bus}+\beta_{access}access_{bus}$$
$$V_{air}=asc_{air}+\beta_{time}time_{air}+\beta_{cost}cost_{air}+\beta_{access}access_{air}+\sum_{l=2}^{3}\beta_{service=l}\mathbb{I}(service_{air}==l)$$
$$V_{rail}=asc_{rail}+\beta_{time}time_{rail}+\beta_{cost}cost_{rail}+\beta_{access}access_{rail}+\sum_{l=2}^{3}\beta_{service=l}\mathbb{I}(service_{rail}==l)$$

Rail serves as the reference alternative ($asc_{rail} = 0$), and no-frills service quality serves as the base level ($b_{service\_1} = 0$).

`3` Report the final log-likelihood of **Model 1**.

In [ ]:
# Your code here

`4` Interpret the estimated parameter values and their significance. Are the signs behaviourally plausible?

In [ ]:
# Your code here

### `Part 3: Error-components logit model with a panel effect`

Estimate **Model 2**, which extends **Model 1** with an error component that captures shared unobserved utility across alternatives and individual-level panel effects.

In labelled choice data, error components are commonly used to capture shared unobserved utility among alternatives belonging to the same nest. Here, **bus** and **rail** are both Public Transport (PT) modes, sharing unobserved characteristics that distinguish them from car and air travel. We therefore specify a shared error component $\upsilon_{n,\text{PT}}$ for the public transport alternatives:

$$V_{car}=asc_{car}+\beta_{time}time_{car}+\beta_{cost}cost_{car}$$
$$V_{bus}=asc_{bus}+\beta_{time}time_{bus}+\beta_{cost}cost_{bus}+\beta_{access}access_{bus} + \upsilon_{n,\text{PT}}$$
$$V_{air}=asc_{air}+\beta_{time}time_{air}+\beta_{cost}cost_{air}+\beta_{access}access_{air}+\sum_{l=2}^{3}\beta_{service=l}\mathbb{I}(service_{air}==l)$$
$$V_{rail}=asc_{rail}+\beta_{time}time_{rail}+\beta_{cost}cost_{rail}+\beta_{access}access_{rail}+\sum_{l=2}^{3}\beta_{service=l}\mathbb{I}(service_{rail}==l)+ \upsilon_{n,\text{PT}}$$

where $\upsilon_{n,\text{PT}} = \sigma_{\upsilon}\eta_{n,\text{PT}}$ with $\eta_{n,\text{PT}} \sim N(0,1)$, such that $\upsilon_{n,\text{PT}} \sim N(0,\sigma_{\upsilon}^2)$, is an individual-specific error component. In your code, define the standard deviation parameter as `sigma_ups` and generate the error component using 500 `NORMAL_HALTON2` draws.

`5` Report the final log-likelihood of **Model 2**.

In [ ]:
# Your code here

`6` Interpret the estimated panel-effect parameter and its significance. Does the result provide evidence of correlation in the unobserved utilities across the repeated choice tasks of the same respondent?

In [ ]:
# Your code here

`7` Explain why bus and rail share the same error component draw, why car and air do not have this error component.

In [ ]:
# Your code here

### `Part 4: Normally distributed taste heterogeneity`

Estimate **Model 3**, which extends **Model 2** by allowing the taste parameter for travel time to vary across respondents according to a normal distribution:

$$\beta_{time,n}=\mu_{time}+\sigma_{time}\eta_{time,n}, \qquad \eta_{time,n}\sim N(0,1).$$

The utility functions are:

$$V_{car}=asc_{car}+\beta_{time,n}time_{car}+\beta_{cost}cost_{car}$$
$$V_{bus}=asc_{bus}+\beta_{time,n}time_{bus}+\beta_{cost}cost_{bus}+\beta_{access}access_{bus} + \upsilon_{n,\text{PT}}$$
$$V_{air}=asc_{air}+\beta_{time,n}time_{air}+\beta_{cost}cost_{air}+\beta_{access}access_{air}+\sum_{l=2}^{3}\beta_{service=l}\mathbb{I}(service_{air}==l)$$
$$V_{rail}=asc_{rail}+\beta_{time,n}time_{rail}+\beta_{cost}cost_{rail}+\beta_{access}access_{rail}+\sum_{l=2}^{3}\beta_{service=l}\mathbb{I}(service_{rail}==l)+ \upsilon_{n,\text{PT}}$$

Retain the `NORMAL_HALTON2` panel draws from Model 2 (`upsilon_PT`), and use `NORMAL_HALTON2` draws for travel time (`eta_time`). Estimate the model using 500 draws.

`8` Report the final log-likelihood of **Model 3**.

In [ ]:
# Your code here

`9` Interpret the estimated mean and standard deviation of the travel time parameter and their significance. What do they suggest about taste heterogeneity?

In [ ]:
# Your code here

### `Part 5: Negative lognormal taste heterogeneity`

Estimate **Model 4**, which extends **Model 2** in the same way as Model 3, but assumes that the travel time coefficient follows a negative lognormal distribution instead of a normal distribution:

$$\beta_{time,n}=-\exp(\mu_{time,log}+\sigma_{time,log}\eta_{time,log,n}), \qquad \eta_{time,log,n}\sim N(0,1).$$

The utility functions are:

$$V_{car}=asc_{car}+\beta_{time,n}time_{car}+\beta_{cost}cost_{car}$$
$$V_{bus}=asc_{bus}+\beta_{time,n}time_{bus}+\beta_{cost}cost_{bus}+\beta_{access}access_{bus} + \upsilon_{n,\text{PT}}$$
$$V_{air}=asc_{air}+\beta_{time,n}time_{air}+\beta_{cost}cost_{air}+\beta_{access}access_{air}+\sum_{l=2}^{3}\beta_{service=l}\mathbb{I}(service_{air}==l)$$
$$V_{rail}=asc_{rail}+\beta_{time,n}time_{rail}+\beta_{cost}cost_{rail}+\beta_{access}access_{rail}+\sum_{l=2}^{3}\beta_{service=l}\mathbb{I}(service_{rail}==l)+ \upsilon_{n,\text{PT}}$$

Retain the `NORMAL_HALTON2` panel draws from Model 2 (`upsilon_PT`), and use `NORMAL` draws for the lognormal travel time parameter (`eta_time_log`). Estimate the model using 500 draws.

`10` Report the final log-likelihood of **Model 4**.

In [ ]:
# Your code here

`11` Which model (**Models 3 or Model 4**) would you report?

In [ ]:
# Your code here

`12` Calculate the mean and median of the negative lognormal travel time coefficient. Interpret their signs and explain why the mean and median differ.

In [ ]:
# Your code here